In [ ]:
!pip install langchain langchain-community langchain-google-genai faiss-cpu gradio pypdf sentence-transformers -q

In [ ]:
import os
import gradio as gr
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("✅ All imports successful!")

✅ All imports successful!


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

In [ ]:
! pip install langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.2 MB/s eta 0:00:00


In [ ]:
# setting your gemini API key
os.environ["GROQ_API_KEY"]= "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"
print("API key SET")

API key SET


In [ ]:
from langchain_groq import ChatGroq

def process_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(documents)
    from langchain_community.embeddings import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    return vectorstore

def ask_question(pdf_file, question, history):
    if pdf_file is None:
        history.append(("", "⚠️ Please upload an insurance PDF first!"))
        return history
    try:
        vectorstore = process_pdf(pdf_file)
        retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
        docs = retriever.invoke(question)
        context = "\n\n".join([d.page_content for d in docs])

        llm = ChatGroq(
            model="llama-3.1-8b-instant",
            temperature=0.2,
            api_key=os.environ["GROQ_API_KEY"]
        )

        prompt = f"""You are InsureBot, an expert AI assistant for insurance policies.
Use the context below to answer the question clearly and simply.
If the answer is not in the document, say "This information is not found in the uploaded policy."

Context: {context}

Question: {question}

Answer:"""

        from langchain_core.messages import HumanMessage
        response = llm.invoke([HumanMessage(content=prompt)])
        answer = response.content
        history.append((question, answer))
        return history

    except Exception as e:
        history.append((question, f"❌ Error: {str(e)}"))
        return history

print("✅ InsureBot logic ready with Groq!")

✅ InsureBot logic ready with Groq!


In [ ]:
! pip install google-genai -q

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="InsureBot") as app:

    gr.Markdown("""
    # 🏥 InsureBot — AI Insurance Policy Assistant
    **Upload any insurance policy PDF → Ask questions → Get instant AI answers**
    """)

    with gr.Row():
        with gr.Column(scale=1):
            pdf_upload = gr.File(
                label="📄 Upload Insurance Policy PDF",
                file_types=[".pdf"]
            )
            gr.Markdown("""
            **Try asking:**
            - What is covered under this policy?
            - What is the claim process?
            - What are the exclusions?
            - What is the sum insured?
            - What is the waiting period?
            """)

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="InsureBot 🤖", height=400)
            question_box = gr.Textbox(
                placeholder="Ask anything about your policy...",
                label="Your Question"
            )
            with gr.Row():
                submit_btn = gr.Button("Ask 🤖", variant="primary")
                clear_btn = gr.Button("Clear 🗑️")

    submit_btn.click(
        fn=ask_question,
        inputs=[pdf_upload, question_box, chatbot],
        outputs=[chatbot]
    )
    question_box.submit(
        fn=ask_question,
        inputs=[pdf_upload, question_box, chatbot],
        outputs=[chatbot]
    )
    clear_btn.click(lambda: [], outputs=[chatbot])

app.launch(share=True)

/tmp/ipykernel_3021/3689743861.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="InsureBot") as app:
/tmp/ipykernel_3021/3689743861.py:24: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="InsureBot 🤖", height=400)
/tmp/ipykernel_3021/3689743861.py:24: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="InsureBot 🤖", height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://14ebe64e6a2eb3b9a4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
